# 4.0: Load Silver + Fact + Dimension Tables

### 1. Load Silver Table

In [0]:
df_silver = spark.table("silver_operational_reporting")


### 2. Load Fact Table

In [0]:
fact_requests = spark.table("fact_requests")


### 3. Load Dimension Tables

In [0]:
dim_department = spark.table("dim_department")
dim_employee   = spark.table("dim_employee")
dim_priority   = spark.table("dim_priority")


### 4. Audit Row Counts
Check counts to ensure all tables loaded correctly:

In [0]:
print("Silver rows:", df_silver.count())
print("Fact rows:", fact_requests.count())
print("Department dimension rows:", dim_department.count())
print("Employee dimension rows:", dim_employee.count())
print("Priority dimension rows:", dim_priority.count())


Silver rows: 10000
Fact rows: 10000
Department dimension rows: 4
Employee dimension rows: 401
Priority dimension rows: 3


# 4.1: SLA Compliance KPI

### Aggregate SLA Status
Count requests by SLA outcome:

In [0]:
from pyspark.sql.functions import count, col

sla_kpi = fact_requests.groupBy("SLA_Status").agg(count("*").alias("Request_Count"))
sla_kpi.show()


+----------+-------------+
|SLA_Status|Request_Count|
+----------+-------------+
|  Breached|         3315|
|       Met|         3322|
|   Unknown|         3363|
+----------+-------------+



### Compute SLA Compliance %
Calculate the percentage of requests that met SLA:

In [0]:
total_requests = fact_requests.count()
sla_met = fact_requests.filter(col("SLA_Status") == "Met").count()

sla_compliance_pct = (sla_met / total_requests) * 100
print("SLA Compliance %:", round(sla_compliance_pct, 2))


SLA Compliance %: 33.22


### . Persist KPI Table
Save results into Delta Lake for BI dashboards:

In [0]:
sla_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_sla_kpi")


### Audit Logging
Verify counts:

In [0]:
print("Total requests:", total_requests)
print("SLA Met requests:", sla_met)
print("SLA Compliance %:", round(sla_compliance_pct, 2))


Total requests: 10000
SLA Met requests: 3322
SLA Compliance %: 33.22


# 4.2: Processing Time KPIs

### Overall Processing Time Metrics
Compute average, minimum, and maximum request durations:

In [0]:
from pyspark.sql.functions import avg, min, max

processing_kpi = fact_requests.agg(
    avg("Processing_Time_Recomputed").alias("Avg_Processing_Time"),
    min("Processing_Time_Recomputed").alias("Min_Processing_Time"),
    max("Processing_Time_Recomputed").alias("Max_Processing_Time")
)

processing_kpi.show()


+-------------------+-------------------+-------------------+
|Avg_Processing_Time|Min_Processing_Time|Max_Processing_Time|
+-------------------+-------------------+-------------------+
|           139.6903|               -1.0|              500.0|
+-------------------+-------------------+-------------------+



### By Department
Break down metrics by department:

In [0]:
dept_processing_kpi = fact_requests.groupBy("Department").agg(
    avg("Processing_Time_Recomputed").alias("Avg_Processing_Time"),
    min("Processing_Time_Recomputed").alias("Min_Processing_Time"),
    max("Processing_Time_Recomputed").alias("Max_Processing_Time")
)

dept_processing_kpi.show()


+----------+-------------------+-------------------+-------------------+
|Department|Avg_Processing_Time|Min_Processing_Time|Max_Processing_Time|
+----------+-------------------+-------------------+-------------------+
|        IT| 141.54593495934958|               -1.0|              500.0|
|        HR| 140.03479923518165|               -1.0|              500.0|
|Operations| 137.57764995891537|               -1.0|              500.0|
|   Finance| 139.56041750301083|               -1.0|              500.0|
+----------+-------------------+-------------------+-------------------+



### By Priority
Analyze how request urgency impacts processing time:

In [0]:
priority_processing_kpi = fact_requests.groupBy("Priority").agg(
    avg("Processing_Time_Recomputed").alias("Avg_Processing_Time"),
    min("Processing_Time_Recomputed").alias("Min_Processing_Time"),
    max("Processing_Time_Recomputed").alias("Max_Processing_Time")
)

priority_processing_kpi.show()


+--------+-------------------+-------------------+-------------------+
|Priority|Avg_Processing_Time|Min_Processing_Time|Max_Processing_Time|
+--------+-------------------+-------------------+-------------------+
|     Low| 141.00735294117646|               -1.0|              500.0|
|  Medium| 139.41850490196077|               -1.0|              500.0|
|    High| 138.61390887290167|               -1.0|              499.0|
+--------+-------------------+-------------------+-------------------+



### Persist KPI Tables
Save results into Delta Lake for BI dashboards:

In [0]:
processing_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_processing_kpi")
dept_processing_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_dept_processing_kpi")
priority_processing_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_priority_processing_kpi")


# 4.3: Departmental SLA KPIs

### Aggregate SLA by Department
Count requests by SLA status for each department:

In [0]:
dept_sla_kpi = fact_requests.groupBy("Department", "SLA_Status") \
    .agg(count("*").alias("Request_Count"))

dept_sla_kpi.show()


+----------+----------+-------------+
|Department|SLA_Status|Request_Count|
+----------+----------+-------------+
|        IT|  Breached|          840|
|        HR|  Breached|          874|
|Operations|  Breached|          751|
|        IT|       Met|          754|
|   Finance|   Unknown|          803|
|   Finance|       Met|          838|
|Operations|       Met|          847|
|        IT|   Unknown|          866|
|        HR|   Unknown|          858|
|   Finance|  Breached|          850|
|        HR|       Met|          883|
|Operations|   Unknown|          836|
+----------+----------+-------------+



### Compute Compliance % per Department
Calculate SLA compliance percentage for each department:

In [0]:
from pyspark.sql.functions import sum as _sum

dept_total = dept_sla_kpi.groupBy("Department") \
    .agg(_sum("Request_Count").alias("Total_Requests"))

dept_sla_compliance = dept_sla_kpi.filter(col("SLA_Status") == "Met") \
    .groupBy("Department") \
    .agg(_sum("Request_Count").alias("SLA_Met"))

dept_sla_result = dept_sla_compliance.join(dept_total, "Department") \
    .withColumn("SLA_Compliance_Pct", (col("SLA_Met") / col("Total_Requests")) * 100)

dept_sla_result.show()


+----------+-------+--------------+------------------+
|Department|SLA_Met|Total_Requests|SLA_Compliance_Pct|
+----------+-------+--------------+------------------+
|        IT|    754|          2460|30.650406504065042|
|   Finance|    838|          2491|33.641107988759536|
|        HR|    883|          2615|33.766730401529635|
|Operations|    847|          2434|34.798685291700906|
+----------+-------+--------------+------------------+



### Persist KPI Table
Save results into Delta Lake for BI dashboards:

In [0]:
dept_sla_result.write.format("delta").mode("overwrite").saveAsTable("gold_dept_sla_kpi")


### Audit Logging
Verify compliance percentages:

In [0]:
for row in dept_sla_result.collect():
    print(f"Department: {row['Department']}, SLA Compliance %: {round(row['SLA_Compliance_Pct'], 2)}")


Department: IT, SLA Compliance %: 30.65
Department: Finance, SLA Compliance %: 33.64
Department: HR, SLA Compliance %: 33.77
Department: Operations, SLA Compliance %: 34.8


# 4.4: Priority‑based SLA KPIs

### Aggregate SLA by Priority
Count requests by SLA status for each priority:

In [0]:
priority_sla_kpi = fact_requests.groupBy("Priority", "SLA_Status") \
    .agg(count("*").alias("Request_Count"))

priority_sla_kpi.show()


+--------+----------+-------------+
|Priority|SLA_Status|Request_Count|
+--------+----------+-------------+
|     Low|  Breached|         1099|
|  Medium|  Breached|         1092|
|  Medium|       Met|         1072|
|  Medium|   Unknown|         1100|
|     Low|       Met|         1144|
|    High|       Met|         1106|
|    High|   Unknown|         1106|
|     Low|   Unknown|         1157|
|    High|  Breached|         1124|
+--------+----------+-------------+



### Compute Compliance % per Priority
Calculate SLA compliance percentage for each priority level:

In [0]:
from pyspark.sql.functions import sum as _sum

priority_total = priority_sla_kpi.groupBy("Priority") \
    .agg(_sum("Request_Count").alias("Total_Requests"))

priority_sla_compliance = priority_sla_kpi.filter(col("SLA_Status") == "Met") \
    .groupBy("Priority") \
    .agg(_sum("Request_Count").alias("SLA_Met"))

priority_sla_result = priority_sla_compliance.join(priority_total, "Priority") \
    .withColumn("SLA_Compliance_Pct", (col("SLA_Met") / col("Total_Requests")) * 100)

priority_sla_result.show()


+--------+-------+--------------+------------------+
|Priority|SLA_Met|Total_Requests|SLA_Compliance_Pct|
+--------+-------+--------------+------------------+
|    High|   1106|          3336| 33.15347721822542|
|  Medium|   1072|          3264| 32.84313725490196|
|     Low|   1144|          3400| 33.64705882352941|
+--------+-------+--------------+------------------+



### Persist KPI Table
Save results into Delta Lake for BI dashboards:

In [0]:
priority_sla_result.write.format("delta").mode("overwrite").saveAsTable("gold_priority_sla_kpi")


### Audit Logging
Verify compliance percentages:

In [0]:
for row in priority_sla_result.collect():
    print(f"Priority: {row['Priority']}, SLA Compliance %: {round(row['SLA_Compliance_Pct'], 2)}")


Priority: High, SLA Compliance %: 33.15
Priority: Medium, SLA Compliance %: 32.84
Priority: Low, SLA Compliance %: 33.65


# 4.5: Combined Department + Priority SLA KPIs

### Aggregate SLA by Department + Priority
Count requests by SLA status for each department/priority combination:

In [0]:
dept_priority_sla_kpi = fact_requests.groupBy("Department", "Priority", "SLA_Status") \
    .agg(count("*").alias("Request_Count"))

dept_priority_sla_kpi.show()


+----------+--------+----------+-------------+
|Department|Priority|SLA_Status|Request_Count|
+----------+--------+----------+-------------+
|        IT|     Low|  Breached|          262|
|        HR|  Medium|  Breached|          284|
|Operations|     Low|  Breached|          244|
|        IT|  Medium|       Met|          230|
|   Finance|  Medium|   Unknown|          264|
|   Finance|  Medium|       Met|          272|
|   Finance|     Low|       Met|          275|
|        IT|    High|       Met|          256|
|Operations|  Medium|       Met|          290|
|        IT|    High|   Unknown|          288|
|        HR|  Medium|   Unknown|          299|
|   Finance|  Medium|  Breached|          289|
|        IT|     Low|   Unknown|          302|
|        IT|  Medium|  Breached|          270|
|Operations|     Low|       Met|          289|
|        HR|     Low|       Met|          312|
|        HR|    High|       Met|          291|
|   Finance|     Low|  Breached|          274|
|        HR| 

### Compute Compliance % per Combination
Calculate SLA compliance percentage for each department/priority pair:

In [0]:
from pyspark.sql.functions import sum as _sum, col

# Total requests per Department + Priority
dept_priority_total = dept_priority_sla_kpi.groupBy("Department", "Priority") \
    .agg(_sum("Request_Count").alias("Total_Requests"))

# SLA Met requests per Department + Priority
dept_priority_sla_met = dept_priority_sla_kpi.filter(col("SLA_Status") == "Met") \
    .groupBy("Department", "Priority") \
    .agg(_sum("Request_Count").alias("SLA_Met"))

# Join and compute compliance %
dept_priority_result = dept_priority_sla_met.join(dept_priority_total, ["Department", "Priority"]) \
    .withColumn("SLA_Compliance_Pct", (col("SLA_Met") / col("Total_Requests")) * 100)

dept_priority_result.show()


+----------+--------+-------+--------------+------------------+
|Department|Priority|SLA_Met|Total_Requests|SLA_Compliance_Pct|
+----------+--------+-------+--------------+------------------+
|Operations|     Low|    289|           838| 34.48687350835322|
|   Finance|  Medium|    272|           825| 32.96969696969697|
|        HR|  Medium|    280|           863|  32.4449594438007|
|        HR|     Low|    312|           903| 34.55149501661129|
|        IT|    High|    256|           852|30.046948356807512|
|        IT|     Low|    268|           832| 32.21153846153847|
|        HR|    High|    291|           849|34.275618374558306|
|        IT|  Medium|    230|           776| 29.63917525773196|
|Operations|    High|    268|           796| 33.66834170854271|
|   Finance|    High|    291|           839| 34.68414779499404|
|Operations|  Medium|    290|           800|             36.25|
|   Finance|     Low|    275|           827|33.252720677146314|
+----------+--------+-------+-----------

### Persist KPI Table
Save results into Delta Lake for BI dashboards:

In [0]:
dept_priority_result.write.format("delta").mode("overwrite").saveAsTable("gold_dept_priority_sla_kpi")


### Audit Logging
Verify compliance percentages:

In [0]:
for row in dept_priority_result.collect():
    print(f"Department: {row['Department']}, Priority: {row['Priority']}, SLA Compliance %: {round(row['SLA_Compliance_Pct'], 2)}")


Department: Finance, Priority: Medium, SLA Compliance %: 32.97
Department: Operations, Priority: Low, SLA Compliance %: 34.49
Department: HR, Priority: Medium, SLA Compliance %: 32.44
Department: HR, Priority: Low, SLA Compliance %: 34.55
Department: IT, Priority: High, SLA Compliance %: 30.05
Department: HR, Priority: High, SLA Compliance %: 34.28
Department: IT, Priority: Low, SLA Compliance %: 32.21
Department: IT, Priority: Medium, SLA Compliance %: 29.64
Department: Operations, Priority: High, SLA Compliance %: 33.67
Department: Finance, Priority: High, SLA Compliance %: 34.68
Department: Operations, Priority: Medium, SLA Compliance %: 36.25
Department: Finance, Priority: Low, SLA Compliance %: 33.25


# 4.6: Trend Analysis KPIs

### Daily SLA Trend
Aggregate SLA compliance by day:

In [0]:
from pyspark.sql.functions import to_date, count, col, sum as _sum

# Add a Date column from Start_Time
fact_requests_with_date = fact_requests.withColumn("Request_Date", to_date(col("Start_Time")))

# SLA counts per day
daily_sla_kpi = fact_requests_with_date.groupBy("Request_Date", "SLA_Status") \
    .agg(count("*").alias("Request_Count"))

# Total requests per day
daily_total = daily_sla_kpi.groupBy("Request_Date") \
    .agg(_sum("Request_Count").alias("Total_Requests"))

# SLA Met per day
daily_sla_met = daily_sla_kpi.filter(col("SLA_Status") == "Met") \
    .groupBy("Request_Date") \
    .agg(_sum("Request_Count").alias("SLA_Met"))

# Compliance %
daily_sla_result = daily_sla_met.join(daily_total, "Request_Date") \
    .withColumn("SLA_Compliance_Pct", (col("SLA_Met") / col("Total_Requests")) * 100)

daily_sla_result.show()


+------------+-------+--------------+------------------+
|Request_Date|SLA_Met|Total_Requests|SLA_Compliance_Pct|
+------------+-------+--------------+------------------+
|  2026-06-08|    122|           353| 34.56090651558073|
|  2026-06-28|     97|           313|30.990415335463258|
|  2026-06-11|    117|           313| 37.38019169329074|
|  2026-06-01|     75|           223|  33.6322869955157|
|  2026-06-15|    102|           319|31.974921630094045|
|  2026-06-30|    113|           346| 32.65895953757225|
|  2026-06-12|     96|           320|              30.0|
|  2026-06-04|    117|           374|31.283422459893046|
|  2026-06-20|    132|           360|36.666666666666664|
|  2026-06-26|    122|           341| 35.77712609970675|
|  2026-06-14|     96|           306|31.372549019607842|
|  2026-06-24|    129|           372| 34.67741935483871|
|  2026-06-06|    109|           336| 32.44047619047619|
|  2026-06-18|    115|           318| 36.16352201257861|
|  2026-06-21|    117|         

### Weekly Processing Time Trend
Track average processing time by week:

In [0]:
from pyspark.sql.functions import weekofyear, avg

weekly_processing_kpi = fact_requests_with_date.groupBy(weekofyear(col("Start_Time")).alias("Week")) \
    .agg(avg("Processing_Time_Recomputed").alias("Avg_Processing_Time"))

weekly_processing_kpi.show()


+----+-------------------+
|Week|Avg_Processing_Time|
+----+-------------------+
|  24|  143.3149293286219|
|  26| 139.64522998296422|
|  27|  143.2036316472114|
|  23| 137.84303579128934|
|  25| 136.85073977371627|
+----+-------------------+



### Persist KPI Tables
Save results into Delta Lake:

In [0]:
daily_sla_result.write.format("delta").mode("overwrite").saveAsTable("gold_daily_sla_kpi")
weekly_processing_kpi.write.format("delta").mode("overwrite").saveAsTable("gold_weekly_processing_kpi")


### Audit Logging
Verify sample outputs:

In [0]:
print("Daily SLA trend sample:")
for row in daily_sla_result.limit(5).collect():
    print(row)

print("Weekly Processing trend sample:")
for row in weekly_processing_kpi.limit(5).collect():
    print(row)


Daily SLA trend sample:
Row(Request_Date=datetime.date(2026, 6, 8), SLA_Met=122, Total_Requests=353, SLA_Compliance_Pct=34.56090651558073)
Row(Request_Date=datetime.date(2026, 6, 28), SLA_Met=97, Total_Requests=313, SLA_Compliance_Pct=30.990415335463258)
Row(Request_Date=datetime.date(2026, 6, 11), SLA_Met=117, Total_Requests=313, SLA_Compliance_Pct=37.38019169329074)
Row(Request_Date=datetime.date(2026, 6, 1), SLA_Met=75, Total_Requests=223, SLA_Compliance_Pct=33.6322869955157)
Row(Request_Date=datetime.date(2026, 6, 15), SLA_Met=102, Total_Requests=319, SLA_Compliance_Pct=31.974921630094045)
Weekly Processing trend sample:
Row(Week=24, Avg_Processing_Time=143.3149293286219)
Row(Week=26, Avg_Processing_Time=139.64522998296422)
Row(Week=27, Avg_Processing_Time=143.2036316472114)
Row(Week=23, Avg_Processing_Time=137.84303579128934)
Row(Week=25, Avg_Processing_Time=136.85073977371627)


# 4.7: Anomaly Detection KPIs

### Daily Processing Time Distribution
Compute average daily processing times:

In [0]:
from pyspark.sql.functions import to_date, avg, col

fact_requests_with_date = fact_requests.withColumn("Request_Date", to_date(col("Start_Time")))

daily_processing = fact_requests_with_date.groupBy("Request_Date") \
    .agg(avg("Processing_Time_Recomputed").alias("Avg_Processing_Time"))

daily_processing.show()


+------------+-------------------+
|Request_Date|Avg_Processing_Time|
+------------+-------------------+
|  2026-06-13|           141.2625|
|  2026-06-22| 161.78618421052633|
|  2026-07-01| 137.77777777777777|
|  2026-06-27| 153.22388059701493|
|  2026-06-02| 126.43801652892562|
|  2026-06-01| 138.15695067264573|
|  2026-06-08|  143.8441926345609|
|  2026-06-21|  134.8242424242424|
|  2026-06-06|  145.9077380952381|
|  2026-06-17|  149.4908536585366|
|  2026-06-26| 133.77126099706746|
|  2026-06-14| 141.58823529411765|
|  2026-06-28| 128.20127795527156|
|  2026-06-25| 137.37125748502993|
|  2026-06-10| 142.03395061728395|
|  2026-06-15|  139.2884012539185|
|  2026-06-05| 142.03790087463557|
|  2026-06-09|  157.3719512195122|
|  2026-06-03|  139.0119760479042|
|  2026-06-23| 130.68481375358166|
+------------+-------------------+
only showing top 20 rows


### Detect Outliers
Use statistical thresholds (e.g., mean ± 2 standard deviations):

In [0]:
from pyspark.sql.functions import mean, stddev

stats = daily_processing.agg(
    mean("Avg_Processing_Time").alias("Mean_PT"),
    stddev("Avg_Processing_Time").alias("StdDev_PT")
).collect()[0]

mean_pt = stats["Mean_PT"]
stddev_pt = stats["StdDev_PT"]

upper_threshold = mean_pt + 2 * stddev_pt
lower_threshold = mean_pt - 2 * stddev_pt

anomalies = daily_processing.filter(
    (col("Avg_Processing_Time") > upper_threshold) |
    (col("Avg_Processing_Time") < lower_threshold)
)

anomalies.show()


+------------+-------------------+
|Request_Date|Avg_Processing_Time|
+------------+-------------------+
|  2026-06-22| 161.78618421052633|
|  2026-06-09|  157.3719512195122|
+------------+-------------------+



### SLA Breach Anomalies
Identify days with unusually high SLA breach percentages:

In [0]:
daily_sla_kpi = fact_requests_with_date.groupBy("Request_Date", "SLA_Status") \
    .agg(count("*").alias("Request_Count"))

daily_total = daily_sla_kpi.groupBy("Request_Date") \
    .agg(_sum("Request_Count").alias("Total_Requests"))

daily_sla_met = daily_sla_kpi.filter(col("SLA_Status") == "Met") \
    .groupBy("Request_Date") \
    .agg(_sum("Request_Count").alias("SLA_Met"))

daily_sla_result = daily_sla_met.join(daily_total, "Request_Date") \
    .withColumn("SLA_Compliance_Pct", (col("SLA_Met") / col("Total_Requests")) * 100)

sla_stats = daily_sla_result.agg(
    mean("SLA_Compliance_Pct").alias("Mean_SLA"),
    stddev("SLA_Compliance_Pct").alias("StdDev_SLA")
).collect()[0]

mean_sla = sla_stats["Mean_SLA"]
stddev_sla = sla_stats["StdDev_SLA"]

sla_anomalies = daily_sla_result.filter(
    (col("SLA_Compliance_Pct") < mean_sla - 2 * stddev_sla) |
    (col("SLA_Compliance_Pct") > mean_sla + 2 * stddev_sla)
)

sla_anomalies.show()


+------------+-------+--------------+------------------+
|Request_Date|SLA_Met|Total_Requests|SLA_Compliance_Pct|
+------------+-------+--------------+------------------+
|  2026-07-01|     38|            99| 38.38383838383838|
+------------+-------+--------------+------------------+



### Persist Anomaly Tables
Save anomaly results for monitoring dashboards:

In [0]:
anomalies.write.format("delta").mode("overwrite").saveAsTable("gold_processing_anomalies")
sla_anomalies.write.format("delta").mode("overwrite").saveAsTable("gold_sla_anomalies")


# 4.8: Forecasting KPIs

### Prepare Time Series Data
We’ll use daily SLA compliance percentages as the forecasting target:

In [0]:
daily_sla_series = daily_sla_result.select("Request_Date", "SLA_Compliance_Pct") \
    .orderBy("Request_Date")


### Convert to Pandas for Forecasting
Forecasting libraries like Prophet/ARIMA work on Pandas DataFrames:

In [0]:
daily_sla_pd = daily_sla_series.toPandas()
daily_sla_pd.columns = ["ds", "y"]  # Prophet expects ds=date, y=value


### Forecast SLA Compliance
Using Facebook Prophet (or ARIMA if preferred):

In [0]:
%pip install prophet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install statsmodels


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 19.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Prepare Time Series Data
Use daily SLA compliance percentages:

In [0]:
daily_sla_series = daily_sla_result.select("Request_Date", "SLA_Compliance_Pct") \
    .orderBy("Request_Date")

daily_sla_pd = daily_sla_series.toPandas()
daily_sla_pd.columns = ["ds", "y"]  # Prophet expects ds=date, y=value


### Forecast SLA Compliance
Using Facebook Prophet (or ARIMA if preferred):

In [0]:
from prophet import Prophet

model = Prophet()
model.fit(daily_sla_pd)

future = model.make_future_dataframe(periods=30)  # forecast next 30 days
forecast = model.predict(future)

forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()


10:40:21 - cmdstanpy - INFO - Chain [1] start processing
10:40:21 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
56,2026-07-27,36.345864,34.009913,38.665759
57,2026-07-28,35.645769,33.190270,37.990099
58,2026-07-29,37.025564,34.637754,39.441171
59,2026-07-30,38.601232,36.063742,40.942675
60,2026-07-31,36.706989,34.203845,39.150372


### Alternative: ARIMA Forecast
If Prophet installation is problematic, use ARIMA:

In [0]:
import statsmodels.api as sm

model_arima = sm.tsa.ARIMA(daily_sla_pd["y"], order=(5,1,0))
model_fit = model_arima.fit()

forecast_arima = model_fit.forecast(steps=30)
print(forecast_arima)


31    35.285273
32    34.786249
33    33.076233
34    32.315624
35    33.987312
36    34.768384
37    35.191825
38    34.791657
39    33.841409
40    33.592424
41    33.776213
42    34.256180
43    34.615255
44    34.518174
45    34.232987
46    33.986211
47    33.961138
48    34.141095
49    34.314805
50    34.362755
51    34.275902
52    34.151282
53    34.099971
54    34.139490
55    34.218280
56    34.266171
57    34.252524
58    34.204262
59    34.166712
60    34.166258
Name: predicted_mean, dtype: float64


### Persist Forecast Tables
Save forecasts into Delta Lake:

In [0]:
spark.createDataFrame(forecast[["ds","yhat","yhat_lower","yhat_upper"]]) \
    .write.format("delta").mode("overwrite").saveAsTable("gold_sla_forecast")


# 4.9: Scenario Analysis KPIs

### 1. Define Scenarios
Examples:

1. Workload Surge → Requests increase by +20%.
2. Staffing Reduction → SLA compliance drops by −10%.
3. Efficiency Gain → Processing time improves by −15%.

### Apply Scenario Multipliers
Adjust forecast outputs:

In [0]:
from pyspark.sql.functions import col

# Base SLA forecast
sla_forecast_df = spark.createDataFrame(forecast[["ds","yhat"]])

# Scenario multipliers
workload_surge = sla_forecast_df.withColumn("Scenario_SLA", col("yhat") * 0.9)   # compliance drops with surge
staffing_reduction = sla_forecast_df.withColumn("Scenario_SLA", col("yhat") * 0.85)
efficiency_gain = sla_forecast_df.withColumn("Scenario_SLA", col("yhat") * 1.1)

workload_surge.show(5)


+-------------------+------------------+------------------+
|                 ds|              yhat|      Scenario_SLA|
+-------------------+------------------+------------------+
|2026-06-01 00:00:00|31.869682712210906|28.682714440989816|
|2026-06-02 00:00:00|31.169587717205147| 28.05262894548463|
|2026-06-03 00:00:00| 32.54938399558492|29.294445596026428|
|2026-06-04 00:00:00|34.125052734500244| 30.71254746105022|
|2026-06-05 00:00:00| 32.23081002490146|29.007729022411315|
+-------------------+------------------+------------------+
only showing top 5 rows


### Scenario Comparison Table
Combine scenarios into one table for dashboards:

In [0]:
scenario_result = sla_forecast_df \
    .withColumn("Workload_Surge", col("yhat") * 0.9) \
    .withColumn("Staffing_Reduction", col("yhat") * 0.85) \
    .withColumn("Efficiency_Gain", col("yhat") * 1.1)

scenario_result.show(5)


+-------------------+------------------+------------------+------------------+------------------+
|                 ds|              yhat|    Workload_Surge|Staffing_Reduction|   Efficiency_Gain|
+-------------------+------------------+------------------+------------------+------------------+
|2026-06-01 00:00:00|31.869682712210906|28.682714440989816| 27.08923030537927|   35.056650983432|
|2026-06-02 00:00:00|31.169587717205147| 28.05262894548463|26.494149559624375|34.286546488925666|
|2026-06-03 00:00:00| 32.54938399558492|29.294445596026428| 27.66697639624718|35.804322395143416|
|2026-06-04 00:00:00|34.125052734500244| 30.71254746105022|29.006294824325206|37.537558007950274|
|2026-06-05 00:00:00| 32.23081002490146|29.007729022411315|27.396188521166238|35.453891027391606|
+-------------------+------------------+------------------+------------------+------------------+
only showing top 5 rows


### Persist Scenario Tables
Save results into Delta Lake:

In [0]:
scenario_result.write.format("delta").mode("overwrite").saveAsTable("gold_scenario_sla_kpi")


# 4.10: Insights & Narrative

### Summarize SLA Performance
1. Overall SLA compliance is X%.
2. Departments with consistently high compliance show strong operational discipline.
3. Priorities reveal whether urgent requests are being met — if high‑priority compliance lags, that’s a red flag.

### Highlight Processing Time Trends
1. Average processing time is Y minutes, with department‑level variation.
2. Weekly trends show whether efficiency is improving or declining.
3. Anomalies (spikes) indicate bottlenecks or unusual workload surges.

### Department + Priority Matrix Insights
1. Certain departments handle high‑priority requests better than others.
2. Matrix view highlights where SLA breaches concentrate — useful for targeted interventions.

### Forecast & Scenario Insights
1. Forecasts project SLA compliance and processing times for the next 30 days.
2. Scenario analysis shows impact of workload surges, staffing changes, or efficiency gains.
3. This helps leadership plan resources and mitigate risks.